# Full Dataset Preprocessing

Preprocessing pipeline for the merged 4-dataset CSV created by `combine_all_datasets.ipynb`.

This notebook follows the same core preprocessing techniques used in the dataset-specific notebooks:
- unified binary target creation
- missing/infinite value checks
- explicit leakage / identifier feature removal
- documented all-missing / single-valued feature removal
- deterministic 70 / 15 / 15 split
- representative-sample fitted preprocessing transformer
- saved processed splits, feature names, and metadata


## 1) Import Libraries

Loads all required packages for data handling, preprocessing pipelines, splitting, and serialization.

In [ ]:
import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

from scipy import sparse
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

pd.set_option('display.max_columns', 200)
pd.set_option('display.max_rows', 50)


## 2) Configure Paths and Core Settings

Defines portable paths, output locations, constants, and validates that the merged input CSV exists before preprocessing starts.

In [ ]:
# ── Paths (portable — Path.cwd() resolves to the notebook's working directory) ─
# Place this notebook inside:  <project_root>/full_dataset/
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive', force_remount=True)


NOTEBOOK_DIR =  Path('/content/drive/MyDrive/MLmodeling/XAI/Datasets/full_dataset')
PROJECT_ROOT     = Path('/content/drive/MyDrive/MLmodeling/XAI')

FULL_DATASET_DIR = NOTEBOOK_DIR            # dataset files sit alongside the notebook
SPLITS_DIR       = PROJECT_ROOT / 'splits' / 'full_dataset'
# All outputs (splits, preprocessor, metadata, feature names) go to one folder
# for consistency with all other dataset preprocessing notebooks.
SPLITS_DIR.mkdir(parents=True, exist_ok=True)

COMBINED_PATH = FULL_DATASET_DIR / 'outputs' / 'all_4_datasets_combined.csv'
META_PATH     = SPLITS_DIR / 'preprocessing_metadata.json'

RANDOM_STATE    = 42
TARGET_COL      = 'target'
PROVENANCE_COLS = ['source_dataset', 'source_file', 'source_row_id']

print('Project root :', PROJECT_ROOT)
print('Combined CSV :', COMBINED_PATH)
print('Splits dir   :', SPLITS_DIR)

if not COMBINED_PATH.exists():
    raise FileNotFoundError(
        f'Combined dataset not found: {COMBINED_PATH}\n'
        'Run full_dataset/combine_all_datasets.ipynb first.'
    )


Mounted at /content/drive
Project root : /content/drive/MyDrive/MLmodeling/XAI
Combined CSV : /content/drive/MyDrive/MLmodeling/XAI/Datasets/full_dataset/outputs/all_4_datasets_combined.csv
Splits dir   : /content/drive/MyDrive/MLmodeling/XAI/splits/full_dataset


## 3) Define Target and Feature-Removal Rules

Creates the unified binary target logic across source datasets and declares columns that must be removed to prevent leakage or poor generalization.

In [ ]:
import numpy as np
import pandas as pd

DROP_ALWAYS = {
    'source_dataset'      : 'Dataset provenance only — not a network feature and leaks source identity',
    'source_file'         : 'File provenance only — not available at prediction time',
    'source_row_id'       : 'Row identifier — not a network feature and not available at prediction time',
    'label'               : 'Original per-dataset label — used only to derive unified binary target',
    'Attack_type'         : 'Original RT-IoT label — used only to derive unified binary target',
    'type'                : 'Original attack-category column — direct leakage after target derivation',
    'attack_cat'          : 'Original attack-category column — direct leakage after target derivation',
    'id'                  : 'Row identifier — not a network feature, not available at prediction time',
    'no'                  : 'Row counter — not a network feature, not available at prediction time',
}

COMPACT_IP_COLS = ['src_ip', 'dst_ip']
COMPACT_SEQ_COLS = ['stcpb', 'dtcpb']
COMPACT_TEXT_COLS = [
    'dns_query', 'ssl_subject', 'ssl_issuer', 'http_uri',
    'http_user_agent', 'http_orig_mime_types', 'http_resp_mime_types', 'weird_addl',
]
DROP_AFTER_COMPACT_FEATURES = COMPACT_IP_COLS + COMPACT_SEQ_COLS + COMPACT_TEXT_COLS

IP_HASH_BUCKETS = 1024
TEXT_HASH_BUCKETS = 2048
SEQ_MODULUS = 65536

def _hash_bucket(series: pd.Series, n_buckets: int) -> pd.Series:
    s = series.astype('string').fillna('__MISSING__')
    h = pd.util.hash_pandas_object(s, index=False).astype('uint64') % np.uint64(n_buckets)
    return h.astype('float32')

def add_compact_features(df: pd.DataFrame) -> pd.DataFrame:
    feat = {}

    for col in COMPACT_IP_COLS:
        if col not in df.columns:
            continue
        s = df[col].astype('string')
        feat[f'{col}_is_private'] = s.str.match(
            r'^(10\.|192\.168\.|172\.(1[6-9]|2[0-9]|3[0-1])\.)', na=False
        ).astype('float32')
        feat[f'{col}_hash_bucket'] = _hash_bucket(s.str.lower().str.strip(), IP_HASH_BUCKETS)

    for col in COMPACT_SEQ_COLS:
        if col not in df.columns:
            continue
        x = pd.to_numeric(df[col], errors='coerce')
        feat[f'{col}_present'] = x.notna().astype('float32')
        feat[f'{col}_log1p'] = np.log1p(x.clip(lower=0)).fillna(0).astype('float32')
        feat[f'{col}_low16'] = (x.fillna(0).astype('float64') % SEQ_MODULUS).astype('float32')

    for col in COMPACT_TEXT_COLS:
        if col not in df.columns:
            continue
        s = df[col].astype('string').str.strip()
        feat[f'{col}_len'] = s.str.len().fillna(0).clip(0, 2048).astype('float32')
        feat[f'{col}_hash_bucket'] = _hash_bucket(s.str.lower(), TEXT_HASH_BUCKETS)

    if not feat:
        return df
    feat_df = pd.DataFrame(feat, index=df.index)
    return pd.concat([df, feat_df], axis=1)

def build_binary_target(df: pd.DataFrame) -> pd.Series:
    """
    Unify the four source datasets into one binary target: Normal=0, Attack=1.

    CICIoT2023 : label == 'BenignTraffic' -> 0, else 1
    RT-IoT2022 : Attack_type in {Thing_Speak, MQTT_Publish, Wipro_bulb} -> 0, else 1
                 (these are the legitimate IoT device communication classes —
                  confirmed from dataset analysis; NOT 'benign'/'normal' strings)
    TON_IoT    : label column is already binary 0/1 — reused directly
    UNSW-NB15  : label column is already binary 0/1 — reused directly
    """
    y   = pd.Series(pd.NA, index=df.index, dtype='Int64')
    src = df['source_dataset'].astype('string')

    # CICIoT2023: string label
    mask_cic = src.eq('CICIoT2023')
    if mask_cic.any() and 'label' in df.columns:
        cic = df.loc[mask_cic, 'label'].astype('string').str.strip()
        y.loc[mask_cic] = (cic != 'BenignTraffic').astype('Int64')

    # RT-IoT2022: Attack_type — normal classes are IoT device labels, NOT text 'benign'
    RT_NORMAL = {'Thing_Speak', 'MQTT_Publish', 'Wipro_bulb'}
    mask_rt = src.eq('RT-IoT2022')
    if mask_rt.any() and 'Attack_type' in df.columns:
        rt = df.loc[mask_rt, 'Attack_type'].astype('string').str.strip()
        y.loc[mask_rt] = (~rt.isin(RT_NORMAL)).astype('Int64')

    # TON_IoT: numeric label reused directly
    mask_ton = src.eq('TON_IoT')
    if mask_ton.any() and 'label' in df.columns:
        y.loc[mask_ton] = pd.to_numeric(
            df.loc[mask_ton, 'label'], errors='coerce'
        ).astype('Int64')

    # UNSW-NB15: numeric label reused directly
    mask_unsw = src.eq('UNSW-NB15')
    if mask_unsw.any() and 'label' in df.columns:
        y.loc[mask_unsw] = pd.to_numeric(
            df.loc[mask_unsw, 'label'], errors='coerce'
        ).astype('Int64')

    return y

print('Constants, compact encoders, and target function defined.')

Constants, compact encoders, and target function defined.


## 4) Discover Schema and Collect Representative Fitting Sample

**Why this approach:**  
The full dataset has 100+ columns with mixed dtypes (object strings, floats, ints).  
`pd.concat()` of all chunks causes dtype-promotion storms that balloon memory to 30-50 GB.  
Using the earliest rows only for the fit sample can also bias preprocessing toward whichever dataset appears first in the merged CSV.  

**Solution: never materialize the full DataFrame and never fit on an order-biased sample.**  
1. Stream the CSV once to discover schema and collect a representative fit sample across all 4 source datasets and both target classes  
2. Fit the preprocessor on that representative sample only  
3. Stream the CSV a second time: clean each chunk, transform it, write directly to the output CSVs  
4. Peak memory at any point: about 1 chunk plus the fit sample, never the full dataset


In [ ]:
def build_binary_target(df):
    """
    Build binary target:
      0 = Normal / Benign
      1 = Attack

    Handles:
      - CICIoT2023 using label
      - RT-IoT2022 using attack_type
      - TON_IoT using type or label
      - UNSW-NB15 using attack_cat or label
    """

    y = pd.Series(pd.NA, index=df.index, dtype="Int64")

    ds = df["source_dataset"].astype(str).str.strip()

    # -----------------------------
    # CICIoT2023
    # -----------------------------
    cic_mask = ds.eq("CICIoT2023")

    if "label" in df.columns:
        cic_label = df["label"].astype(str).str.strip().str.lower()

        y.loc[cic_mask & cic_label.isin(["benigntraffic", "benign", "normal", "0"])] = 0
        y.loc[cic_mask & ~cic_label.isin(["benigntraffic", "benign", "normal", "0", "nan", "<na>", ""])] = 1

    # -----------------------------
    # RT-IoT2022
    # -----------------------------
    rt_mask = ds.eq("RT-IoT2022")

    if "attack_type" in df.columns:
        rt_label = df["attack_type"].astype(str).str.strip().str.lower()

        # These are normal/benign RT-IoT traffic classes
        rt_benign = {
            "mqtt_publish",
            "thing_speak",
            "wipro_bulb",
            "normal",
            "benign",
            "0"
        }

        y.loc[rt_mask & rt_label.isin(rt_benign)] = 0
        y.loc[rt_mask & ~rt_label.isin(rt_benign | {"nan", "<na>", ""})] = 1

    # -----------------------------
    # TON_IoT
    # -----------------------------
    ton_mask = ds.eq("TON_IoT")

    if "type" in df.columns:
        ton_label = df["type"].astype(str).str.strip().str.lower()

        y.loc[ton_mask & ton_label.isin(["normal", "benign", "0"])] = 0
        y.loc[ton_mask & ~ton_label.isin(["normal", "benign", "0", "nan", "<na>", ""])] = 1

    elif "label" in df.columns:
        ton_label = df["label"].astype(str).str.strip().str.lower()

        y.loc[ton_mask & ton_label.isin(["normal", "benign", "0"])] = 0
        y.loc[ton_mask & ~ton_label.isin(["normal", "benign", "0", "nan", "<na>", ""])] = 1

    # -----------------------------
    # UNSW-NB15
    # -----------------------------
    unsw_mask = ds.eq("UNSW-NB15")

    if "attack_cat" in df.columns:
        unsw_label = df["attack_cat"].astype(str).str.strip().str.lower()

        y.loc[unsw_mask & unsw_label.isin(["normal", "benign", "0"])] = 0
        y.loc[unsw_mask & ~unsw_label.isin(["normal", "benign", "0", "nan", "<na>", ""])] = 1

    elif "label" in df.columns:
        unsw_label = df["label"].astype(str).str.strip().str.lower()

        y.loc[unsw_mask & unsw_label.isin(["0", "normal", "benign"])] = 0
        y.loc[unsw_mask & unsw_label.isin(["1"])] = 1

    return y

In [ ]:
import csv, gc
from collections import Counter
import numpy as np
import pandas as pd

CHUNK_SIZE   = 100_000   # rows per chunk
SAMPLE_SIZE  = 200_000   # rows used to fit the preprocessor
RANDOM_STATE = 42

with open(COMBINED_PATH, 'r', encoding='utf-8-sig', newline='') as _f:
    ALL_CSV_COLS = next(csv.reader(_f))

REQUIRED_AT_LOAD = {'source_dataset', 'label', 'attack_type', 'type', 'attack_cat'}
SKIP_AT_LOAD = {c for c in DROP_ALWAYS if c not in REQUIRED_AT_LOAD and c in ALL_CSV_COLS}
LOAD_COLS = [c for c in ALL_CSV_COLS if c not in SKIP_AT_LOAD]

SOURCE_DATASETS = ['CICIoT2023', 'RT-IoT2022', 'TON_IoT', 'UNSW-NB15']
TARGET_VALUES   = [0, 1]
SAMPLE_BUCKET_TARGET = SAMPLE_SIZE // (len(SOURCE_DATASETS) * len(TARGET_VALUES))

print(f'Total CSV columns       : {len(ALL_CSV_COLS)}')
print(f'Skipped at load (leak)  : {len(SKIP_AT_LOAD)}')
print(f'Columns loaded per chunk: {len(LOAD_COLS)}')
print(f'Per dataset/target quota: {SAMPLE_BUCKET_TARGET:,}')

print(f'\nStreaming pass 1: collecting up to {SAMPLE_SIZE:,} representative fit rows...')
sample_buffers = {(ds, tgt): [] for ds in SOURCE_DATASETS for tgt in TARGET_VALUES}
bucket_counts  = Counter({(ds, tgt): 0 for ds in SOURCE_DATASETS for tgt in TARGET_VALUES})
total_raw_rows   = 0
total_valid_rows = 0

for chunk in pd.read_csv(
    COMBINED_PATH,
    usecols=LOAD_COLS,
    chunksize=CHUNK_SIZE,
    engine='python',
    on_bad_lines='skip',
    encoding='utf-8-sig',
):
    total_raw_rows += len(chunk)

    tgt = build_binary_target(chunk)
    valid = tgt.notna()
    chunk = chunk.loc[valid].copy()
    tgt   = tgt.loc[valid].astype(int)
    total_valid_rows += len(chunk)
    if chunk.empty:
        continue

    source_series = chunk['source_dataset'].astype('string').copy()

    # Keep signal from high-cardinality/raw fields using bounded numeric encodings.
    chunk = add_compact_features(chunk)

    for col in chunk.select_dtypes(include=[np.number]).columns:
        if col in (TARGET_COL, 'label'):
            continue
        if pd.api.types.is_float_dtype(chunk[col]):
            chunk[col] = pd.to_numeric(chunk[col], downcast='float')
        else:
            chunk[col] = pd.to_numeric(chunk[col], downcast='integer')

    drop_now = [c for c in DROP_ALWAYS if c in chunk.columns and c not in REQUIRED_AT_LOAD]
    chunk.drop(columns=drop_now, inplace=True)
    drop_after_target = [c for c in (
        ['source_dataset', 'source_file', 'source_row_id', 'label', 'attack_type', 'Attack_type', 'type', 'attack_cat']
        + DROP_AFTER_COMPACT_FEATURES
    ) if c in chunk.columns]
    chunk.drop(columns=drop_after_target, inplace=True)
    chunk[TARGET_COL] = tgt.values

    for ds in SOURCE_DATASETS:
        ds_mask = source_series.eq(ds)
        if not ds_mask.any():
            continue
        for tgt_value in TARGET_VALUES:
            bucket_key = (ds, tgt_value)
            need = SAMPLE_BUCKET_TARGET - bucket_counts[bucket_key]
            if need <= 0:
                continue
            bucket_rows = chunk.loc[ds_mask & chunk[TARGET_COL].eq(tgt_value)]
            if bucket_rows.empty:
                continue
            taken = bucket_rows.iloc[:need].copy()
            sample_buffers[bucket_key].append(taken)
            bucket_counts[bucket_key] += len(taken)

all_buckets_full = all(bucket_counts[k] >= SAMPLE_BUCKET_TARGET for k in sample_buffers)
if all_buckets_full:
    print(f'  Representative sample quotas filled after scanning {total_raw_rows:,} raw rows.')
else:
    print(f'  Stream complete with some underfilled buckets after {total_raw_rows:,} raw rows.')

sample_parts = []
for ds in SOURCE_DATASETS:
    for tgt_value in TARGET_VALUES:
        bucket_key = (ds, tgt_value)
        if sample_buffers[bucket_key]:
            sample_parts.extend(sample_buffers[bucket_key])

if not sample_parts:
    raise ValueError('Representative fit sample is empty; cannot continue preprocessing.')

sample_df = pd.concat(sample_parts, ignore_index=True)
sample_df = sample_df.sample(frac=1.0, random_state=RANDOM_STATE).reset_index(drop=True)
del sample_parts, sample_buffers
gc.collect()

print(f'Fitting sample shape : {sample_df.shape}')
print(f'  Valid rows seen    : {total_valid_rows:,}')
print(f'  Normal (0)         : {(sample_df[TARGET_COL] == 0).sum():,}')
print(f'  Attack (1)         : {(sample_df[TARGET_COL] == 1).sum():,}')
print(f'  Memory             : {sample_df.memory_usage(deep=True).sum()/1e6:.1f} MB')
print('  Sample rows per (dataset, target):')
for ds in SOURCE_DATASETS:
    counts = {tgt_value: bucket_counts[(ds, tgt_value)] for tgt_value in TARGET_VALUES}
    print(f'    {ds}: {counts}')

FEATURE_COLS = [c for c in sample_df.columns if c != TARGET_COL]
print(f'\nFeature columns after leakage drops: {len(FEATURE_COLS)}')


Total CSV columns       : 215
Skipped at load (leak)  : 4
Columns loaded per chunk: 211
Per dataset/target quota: 25,000

Streaming pass 1: collecting up to 200,000 representative fit rows...


/tmp/ipykernel_8032/3691098139.py:70: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  chunk[TARGET_COL] = tgt.values
/tmp/ipykernel_8032/3691098139.py:70: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  chunk[TARGET_COL] = tgt.values
/tmp/ipykernel_8032/3691098139.py:70: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = fra

  Stream complete with some underfilled buckets after 8,437,506 raw rows.
Fitting sample shape : (187507, 221)
  Valid rows seen    : 8,437,506
  Normal (0)         : 87,507
  Attack (1)         : 100,000
  Memory             : 313.8 MB
  Sample rows per (dataset, target):
    CICIoT2023: {0: 25000, 1: 25000}
    RT-IoT2022: {0: 12507, 1: 25000}
    TON_IoT: {0: 25000, 1: 25000}
    UNSW-NB15: {0: 25000, 1: 25000}

Feature columns after leakage drops: 220


## 5) Fit Preprocessor on Representative Sample

Fit is performed on the representative sample only, never on the full dataset.
Feature removal is allowed only with a clear reason, such as leakage, all-missing values, or a single non-null value in the representative sample.


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

X_sample = sample_df.drop(columns=[TARGET_COL])
y_sample = sample_df[TARGET_COL]

numeric_features     = X_sample.select_dtypes(include=[np.number, 'bool']).columns.tolist()
categorical_features = [c for c in X_sample.columns if c not in numeric_features]

print(f'Numeric features     : {len(numeric_features)}')
print(f'Categorical features : {len(categorical_features)}')
if categorical_features:
    print(f'  -> {categorical_features}')

non_null_counts = X_sample.notna().sum()
all_missing_cols = [c for c in X_sample.columns if non_null_counts[c] == 0]
constant_cols = [
    c for c in numeric_features + categorical_features
    if non_null_counts[c] > 0 and X_sample[c].nunique(dropna=True) <= 1
]
cols_to_remove = all_missing_cols + [c for c in constant_cols if c not in all_missing_cols]
if cols_to_remove:
    print('\nColumns removed before fit:')
    if all_missing_cols:
        print(f'  Completely missing in representative sample: {all_missing_cols}')
    if constant_cols:
        print(f'  Single-valued in representative sample     : {constant_cols}')
    X_sample.drop(columns=cols_to_remove, inplace=True)
    numeric_features     = [c for c in numeric_features     if c not in cols_to_remove]
    categorical_features = [c for c in categorical_features if c not in cols_to_remove]
    FEATURE_COLS         = [c for c in FEATURE_COLS         if c not in cols_to_remove]
    for c in all_missing_cols:
        DROP_ALWAYS[c] = 'Representative sample found only missing values for this feature'
    for c in constant_cols:
        DROP_ALWAYS[c] = 'Representative sample found only one non-null value for this feature'
else:
    print('No all-missing or constant columns found in representative sample.')
## fill null with the median and for non numeric fill with the most frequent

numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler()),
])
categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot',  OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
])
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_pipeline,     numeric_features),
        ('cat', categorical_pipeline, categorical_features),
    ],
    remainder='drop'
)

preprocessor.fit(X_sample)
print('\nPreprocessor fitted on representative sample.')

ohe_names = []
if categorical_features:
    ohe_names = list(
        preprocessor.named_transformers_['cat']
        .named_steps['onehot']
        .get_feature_names_out(categorical_features)
    )
feature_names_out = numeric_features + ohe_names
print(f'Output features after OHE: {len(feature_names_out)}')

import joblib
joblib.dump(preprocessor, SPLITS_DIR / 'preprocessor.joblib')
pd.Series(feature_names_out).to_csv(SPLITS_DIR / 'feature_names.csv',
                                     index=False, header=False)
print('Preprocessor and feature names saved.')

del sample_df, X_sample, y_sample
gc.collect()
print('Sample freed from memory.')


Numeric features     : 203
Categorical features : 18
  -> ['proto', 'service', 'attack_type', 'conn_state', 'dns_aa', 'dns_rd', 'dns_ra', 'dns_rejected', 'ssl_version', 'ssl_cipher', 'ssl_resumed', 'ssl_established', 'http_trans_depth', 'http_method', 'http_version', 'weird_name', 'weird_notice', 'state']

Columns removed before fit:
  Completely missing in representative sample: ['id.orig_p', 'id.resp_p', 'fwd_pkts_tot', 'bwd_pkts_tot', 'fwd_data_pkts_tot', 'bwd_data_pkts_tot', 'fwd_pkts_per_sec', 'bwd_pkts_per_sec', 'flow_pkts_per_sec', 'down_up_ratio', 'fwd_header_size_tot', 'fwd_header_size_min', 'fwd_header_size_max', 'bwd_header_size_tot', 'bwd_header_size_min', 'bwd_header_size_max', 'flow_fin_flag_count', 'flow_syn_flag_count', 'flow_rst_flag_count', 'fwd_psh_flag_count', 'bwd_psh_flag_count', 'flow_ack_flag_count', 'fwd_urg_flag_count', 'bwd_urg_flag_count', 'flow_cwr_flag_count', 'flow_ece_flag_count', 'fwd_pkts_payload.min', 'fwd_pkts_payload.max', 'fwd_pkts_payload.tot', 'f

## 6) Stream Pass 2 - Clean, Transform, Write to Disk

Each chunk is cleaned, schema-aligned, transformed by the fitted preprocessor,  
and written directly to the output CSVs.  
Peak memory at any point is **1 chunk (~100k rows)** plus transformer state, and the full dataset is never in RAM.

**Row-retention rule:** rows are kept unless there is a documented reason to exclude them.  
In this notebook the only exclusion is a row whose unified target cannot be derived.

**Split assignment:** rows are assigned to train/val/test using deterministic modulo-100 assignment on the global row counter,  
so the 70/15/15 ratio is maintained across chunks without ever holding all splits in memory.


In [ ]:

def assign_split(global_idx: np.ndarray) -> np.ndarray:
    """
    Returns array of 'train'/'val'/'test' strings for each row.
    Bins 0-69 -> train, 70-84 -> val, 85-99 -> test (modulo 100).
    """
    bins = global_idx % 100
    result = np.full(len(global_idx), 'train', dtype=object)
    result[bins >= 70] = 'val'
    result[bins >= 85] = 'test'
    return result

out_paths = {
    'X_train': SPLITS_DIR / 'X_train.csv',
    'X_val'  : SPLITS_DIR / 'X_val.csv',
    'X_test' : SPLITS_DIR / 'X_test.csv',
    'y_train': SPLITS_DIR / 'y_train.csv',
    'y_val'  : SPLITS_DIR / 'y_val.csv',
    'y_test' : SPLITS_DIR / 'y_test.csv',
}
for p in out_paths.values():
    p.unlink(missing_ok=True)

headers_written = {k: False for k in out_paths}

global_row_idx       = 0
rows_written         = {'train': 0, 'val': 0, 'test': 0}
rows_invalid_target  = 0
chunk_num            = 0

print('Streaming pass 2: clean ? transform ? write...')
print(f'Output: {SPLITS_DIR}')
print()

for chunk in pd.read_csv(
    COMBINED_PATH,
    usecols=LOAD_COLS,
    chunksize=CHUNK_SIZE,
    engine='python',
    on_bad_lines='skip',
    encoding='utf-8-sig',
):
    chunk_num += 1

    tgt   = build_binary_target(chunk)
    valid = tgt.notna()
    rows_invalid_target += int((~valid).sum())
    chunk = chunk.loc[valid].copy()
    tgt   = tgt.loc[valid].astype(np.int8)
    if chunk.empty:
        continue

    # Apply the same compact encoders used during fit-sample collection.
    chunk = add_compact_features(chunk)

    drop_now = [c for c in (
        list(DROP_ALWAYS.keys()) + DROP_AFTER_COMPACT_FEATURES +
        ['source_dataset', 'source_file', 'source_row_id', 'label', 'Attack_type', 'type', 'attack_cat']
    ) if c in chunk.columns]
    chunk.drop(columns=drop_now, inplace=True)

    missing_cols = [c for c in FEATURE_COLS if c not in chunk.columns]
    for mc in missing_cols:
        chunk[mc] = np.nan
    chunk = chunk[FEATURE_COLS]

    for col in numeric_features:
        if col not in chunk.columns:
            continue
        if pd.api.types.is_float_dtype(chunk[col]):
            chunk[col] = pd.to_numeric(chunk[col], downcast='float')
        elif pd.api.types.is_integer_dtype(chunk[col]):
            chunk[col] = pd.to_numeric(chunk[col], downcast='integer')

    num_in_chunk = chunk.select_dtypes(include=[np.number]).columns.tolist()
    if num_in_chunk:
        chunk[num_in_chunk] = chunk[num_in_chunk].replace([np.inf, -np.inf], np.nan)

    X_proc = preprocessor.transform(chunk)
    X_proc_df = pd.DataFrame(X_proc, columns=feature_names_out)

    idx    = np.arange(global_row_idx, global_row_idx + len(chunk))
    splits = assign_split(idx)
    global_row_idx += len(chunk)

    for split_name in ('train', 'val', 'test'):
        mask = splits == split_name
        if not mask.any():
            continue
        X_slice = X_proc_df.loc[mask]
        y_slice = tgt.iloc[np.where(mask)[0]]

        write_header_X = not headers_written[f'X_{split_name}']
        write_header_y = not headers_written[f'y_{split_name}']

        X_slice.to_csv(out_paths[f'X_{split_name}'],
                       mode='a', header=write_header_X, index=False)
        y_slice.to_csv(out_paths[f'y_{split_name}'],
                       mode='a', header=write_header_y, index=False)

        headers_written[f'X_{split_name}'] = True
        headers_written[f'y_{split_name}'] = True
        rows_written[split_name] += int(mask.sum())

    if chunk_num % 10 == 0:
        total_written = sum(rows_written.values())
        print(f'  Chunk {chunk_num:4d} | written: {total_written:>8,} '
              f'(train={rows_written["train"]:,} '
              f'val={rows_written["val"]:,} '
              f'test={rows_written["test"]:,})')

    del chunk, X_proc, X_proc_df, tgt, splits
    if chunk_num % 20 == 0:
        gc.collect()

total_written = sum(rows_written.values())
print()
print('=== Stream complete ===')
print(f'Total rows written          : {total_written:>10,}')
print(f'  Train                     : {rows_written["train"]:>10,} ({rows_written["train"]/total_written*100:.1f}%)')
print(f'  Val                       : {rows_written["val"]:>10,} ({rows_written["val"]/total_written*100:.1f}%)')
print(f'  Test                      : {rows_written["test"]:>10,} ({rows_written["test"]/total_written*100:.1f}%)')
print(f'Rows excluded (bad target)  : {rows_invalid_target:>10,}')


Streaming pass 2: clean ? transform ? write...
Output: /content/drive/MyDrive/MLmodeling/XAI/splits/full_dataset

  Chunk   10 | written: 1,000,000 (train=700,000 val=150,000 test=150,000)
  Chunk   20 | written: 2,000,000 (train=1,400,000 val=300,000 test=300,000)
  Chunk   30 | written: 3,000,000 (train=2,100,000 val=450,000 test=450,000)
  Chunk   40 | written: 4,000,000 (train=2,800,000 val=600,000 test=600,000)
  Chunk   50 | written: 5,000,000 (train=3,500,000 val=750,000 test=750,000)
  Chunk   60 | written: 6,000,000 (train=4,200,000 val=900,000 test=900,000)
  Chunk   70 | written: 7,000,000 (train=4,900,000 val=1,050,000 test=1,050,000)
  Chunk   80 | written: 8,000,000 (train=5,600,000 val=1,200,000 test=1,200,000)


## 7) Run Sanity Validation Checks

Verifies there are no NaN/Inf values, feature dimensions are consistent, rows match labels, and scaling behaved as expected.

In [ ]:
# Load small head samples from the written CSVs for verification
# No need to load full files — just check first 2000 rows of each
PREVIEW = 2000
X_train_head = pd.read_csv(SPLITS_DIR / 'X_train.csv', nrows=PREVIEW)
X_val_head   = pd.read_csv(SPLITS_DIR / 'X_val.csv',   nrows=PREVIEW)
X_test_head  = pd.read_csv(SPLITS_DIR / 'X_test.csv',  nrows=PREVIEW)

print('=== Check 1: No NaN in processed arrays (preview) ===')
for name, df_h in [('train', X_train_head), ('val', X_val_head), ('test', X_test_head)]:
    n = int(df_h.isnull().sum().sum())
    print(f'  {name}: {"clean" if n == 0 else f"{n} NaNs found"}')

print()
print('=== Check 2: Feature count consistency ===')
shapes = [X_train_head.shape[1], X_val_head.shape[1], X_test_head.shape[1]]
print(f'  train={shapes[0]}, val={shapes[1]}, test={shapes[2]}')
print(f'  Consistent: {"YES" if len(set(shapes)) == 1 else "MISMATCH"}')

print()
print('=== Check 3: Train means ~ 0 (StandardScaler applied correctly) ===')
for feat in feature_names_out[:5]:
    if feat in X_train_head.columns:
        print(f'  {feat}: {X_train_head[feat].mean():.6f}')

print()
print('=== Check 4: Split row counts ===')
for split_name, cnt in rows_written.items():
    pct = cnt / total_written * 100
    print(f'  {split_name:5s}: {cnt:>10,}  ({pct:.1f}%)')

del X_train_head, X_val_head, X_test_head
gc.collect()


## 8) Print Final Pipeline Summary

Creates a final summary report of row counts, explicit exclusion reasons, feature counts, leakage controls, and saved artifact locations for downstream modeling notebooks.


In [ ]:
import json as _json

metadata = {
    'dataset'               : 'full_dataset (4 merged)',
    'source_datasets'       : ['CICIoT2023', 'RT-IoT2022', 'TON_IoT', 'UNSW-NB15'],
    'combined_csv'          : str(COMBINED_PATH),
    'total_rows_written'    : total_written,
    'rows_excluded_invalid_target': rows_invalid_target,
    'rows_deduplicated'     : 0,
    'train_size'            : rows_written['train'],
    'val_size'              : rows_written['val'],
    'test_size'             : rows_written['test'],
    'n_numeric_features'    : len(numeric_features),
    'n_categorical_features': len(categorical_features),
    'n_features_after_ohe'  : len(feature_names_out),
    'numeric_features'      : numeric_features,
    'categorical_features'  : categorical_features,
    'removed_features'      : {k: v for k, v in DROP_ALWAYS.items()},
    'all_missing_cols_removed': all_missing_cols,
    'constant_cols_removed' : constant_cols,
    'random_state'          : RANDOM_STATE,
    'split_method'          : 'deterministic modulo-100 row index (70/15/15)',
    'preprocessor_fit_on'   : f'representative cross-dataset sample (target {SAMPLE_SIZE:,} rows)',
    'label_map'             : {'0': 'Normal', '1': 'Attack'},
    'target_derivation': {
        'CICIoT2023' : "label == 'BenignTraffic' -> 0, else 1",
        'RT-IoT2022' : "Attack_type in {Thing_Speak, MQTT_Publish, Wipro_bulb} -> 0, else 1",
        'TON_IoT'    : 'numeric label reused directly (already 0/1)',
        'UNSW-NB15'  : 'numeric label reused directly (already 0/1)',
    }
}

with open(META_PATH, 'w', encoding='utf-8') as f:
    _json.dump(metadata, f, indent=2)

print('=' * 62)
print('     FULL DATASET ? PREPROCESSING SUMMARY')
print('=' * 62)
print(f'  Total rows written          : {total_written:>10,}')
print(f'  Rows excluded (bad target)  : {rows_invalid_target:>10,}')
print(f'  Rows deduplicated           : {0:>10,}')
print(f'  SPLITS (modulo-100 assign):')
print(f'    Train : {rows_written["train"]:>10,}  ({rows_written["train"]/total_written*100:.1f}%)')
print(f'    Val   : {rows_written["val"]:>10,}  ({rows_written["val"]/total_written*100:.1f}%)')
print(f'    Test  : {rows_written["test"]:>10,}  ({rows_written["test"]/total_written*100:.1f}%)')
print(f'  Numeric features            : {len(numeric_features)}')
print(f'  Categorical features        : {len(categorical_features)}')
print(f'  Features after OHE          : {len(feature_names_out)}')
print()
print('  LEAKAGE / DROP CONTROLS:')
print('    Preprocessor fitted on representative sample only')
print('    Features dropped only for explicit reasons (leakage, all-missing, or single-valued)')
print('    All label/ID/provenance columns dropped per chunk before transform')
print()
print('  SAVED:')
for name, p in out_paths.items():
    print(f'    {p}')
print(f'    {SPLITS_DIR / "preprocessor.joblib"}')
print(f'    {SPLITS_DIR / "feature_names.csv"}')
print(f'    {META_PATH}')
print('=' * 62)
